In [1]:
import random
from dataclasses import dataclass
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
random.seed(SEED)

# (eventSource, eventName, weight) per role: what each role normally does
ROLE_ACTIONS = {
    "developer": [("ec2.amazonaws.com", "RunInstances", 3), ("ec2.amazonaws.com", "DescribeInstances", 5),
                  ("s3.amazonaws.com", "PutObject", 6), ("s3.amazonaws.com", "GetObject", 6)],
    "admin":     [("iam.amazonaws.com", "ListUsers", 4), ("iam.amazonaws.com", "CreateUser", 1),
                  ("iam.amazonaws.com", "AttachUserPolicy", 1), ("iam.amazonaws.com", "GetUser", 4)],
    "analyst":   [("s3.amazonaws.com", "GetObject", 8), ("athena.amazonaws.com", "StartQueryExecution", 4),
                  ("glue.amazonaws.com", "GetTable", 3)],
    "devops":    [("ec2.amazonaws.com", "DescribeInstances", 5), ("cloudwatch.amazonaws.com", "GetMetricData", 6),
                  ("ec2.amazonaws.com", "StopInstances", 1), ("ec2.amazonaws.com", "StartInstances", 1)],
}

@dataclass
class User:
    name: str
    role: str
    start_hour: int      # usual shift start (UTC)
    end_hour: int        # usual shift end (UTC)
    ip_prefix: str       # usual IP range
    region: str
    events_per_hour: float
    user_agent: str

def make_users(n=30):
    roles = list(ROLE_ACTIONS)
    regions = ["us-east-1", "ap-south-1", "eu-west-1"]
    agents = ["aws-cli/2.15", "console.amazonaws.com", "Boto3/1.34"]
    users = []
    for i in range(n):
        start = int(rng.integers(3, 12))
        users.append(User(
            name=f"user{i:02d}", role=roles[i % len(roles)],
            start_hour=start, end_hour=start + int(rng.integers(7, 10)),
            ip_prefix=f"10.{int(rng.integers(0, 250))}.{int(rng.integers(0, 250))}",
            region=random.choice(regions),
            events_per_hour=float(rng.uniform(2, 8)),
            user_agent=random.choice(agents),
        ))
    return users

users = make_users(30)
pd.DataFrame([u.__dict__ for u in users]).head(8)

,name,role,start_hour,end_hour,ip_prefix,region,events_per_hour,user_agent
0,user00,developer,3,12,10.163.109,eu-west-1,7.151588,aws-cli/2.15
1,user01,admin,3,12,10.50.23,us-east-1,7.853734,Boto3/1.34
2,user02,analyst,9,18,10.179.196,ap-south-1,2.768682,aws-cli/2.15
3,user03,devops,10,18,10.125.92,us-east-1,7.560590,aws-cli/2.15
4,user04,developer,10,18,10.100.205,eu-west-1,4.660485,aws-cli/2.15
5,user05,admin,7,14,10.23.138,eu-west-1,2.382904,Boto3/1.34
6,user06,analyst,10,19,10.69.157,eu-west-1,6.548526,aws-cli/2.15
7,user07,devops,9,17,10.16.242,eu-west-1,7.358727,console.amazonaws.com


In [2]:
from datetime import datetime, timedelta, timezone

START = datetime(2026, 2, 1, tzinfo=timezone.utc)
DAYS = 30
ip_pool = {u.name: [f"{u.ip_prefix}.{int(x)}" for x in rng.integers(1, 255, 3)] for u in users}

def make_event(u, ts, source, name, ip, region=None, error=None):
    return {
        "eventTime": ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "eventSource": source,
        "eventName": name,
        "awsRegion": region or u.region,
        "sourceIPAddress": ip,
        "userAgent": u.user_agent,
        "userIdentity": {"type": "IAMUser", "userName": u.name},
        "errorCode": error,
        "is_anomaly": 0,
        "attack_type": "normal",
    }

def sample_normal(u):
    actions = ROLE_ACTIONS[u.role]
    w = np.array([a[2] for a in actions], dtype=float)
    w /= w.sum()
    events = []
    for d in range(DAYS):
        day = START + timedelta(days=d)
        if day.weekday() >= 5 and rng.random() < 0.9:   # mostly skip weekends
            continue
        for h in range(u.start_hour, u.end_hour):
            for _ in range(rng.poisson(u.events_per_hour)):
                ts = day + timedelta(hours=h, seconds=int(rng.integers(0, 3600)))
                src, name, _ = actions[rng.choice(len(actions), p=w)]
                err = "AccessDenied" if rng.random() < 0.02 else None
                events.append(make_event(u, ts, src, name, random.choice(ip_pool[u.name]), error=err))
    return events

normal = [e for u in users for e in sample_normal(u)]
df_normal = pd.DataFrame(normal).sort_values("eventTime").reset_index(drop=True)
print(df_normal.shape)
print(df_normal.groupby(df_normal.userIdentity.apply(lambda x: x["userName"])).size().describe())
df_normal.head()

(26947, 10)
count      30.000000
mean      898.233333
std       338.842012
min       358.000000
25%       563.500000
50%       925.000000
75%      1146.000000
max      1534.000000
dtype: float64


,eventTime,eventSource,eventName,awsRegion,sourceIPAddress,userAgent,userIdentity,errorCode,is_anomaly,attack_type
0,2026-02-01T06:00:58Z,s3.amazonaws.com,PutObject,us-east-1,10.26.22.148,console.amazonaws.com,"{'type': 'IAMUser', 'userName': 'user28'}",None,0,normal
1,2026-02-01T06:05:52Z,s3.amazonaws.com,GetObject,ap-south-1,10.163.117.36,aws-cli/2.15,"{'type': 'IAMUser', 'userName': 'user20'}",None,0,normal
2,2026-02-01T06:16:05Z,ec2.amazonaws.com,DescribeInstances,us-east-1,10.26.22.148,console.amazonaws.com,"{'type': 'IAMUser', 'userName': 'user28'}",None,0,normal
3,2026-02-01T06:17:03Z,s3.amazonaws.com,GetObject,ap-south-1,10.163.117.25,aws-cli/2.15,"{'type': 'IAMUser', 'userName': 'user20'}",None,0,normal
4,2026-02-01T06:17:43Z,s3.amazonaws.com,GetObject,us-east-1,10.26.22.148,console.amazonaws.com,"{'type': 'IAMUser', 'userName': 'user28'}",None,0,normal


In [3]:
user_by_name = {u.name: u for u in users}
SUSPICIOUS_IPS = [f"203.0.113.{i}" for i in range(1, 255)]
EXOTIC_REGIONS = ["ap-southeast-1", "sa-east-1", "af-south-1", "eu-north-1"]

def ts_at(day, hour):
    return START + timedelta(days=day, hours=hour, seconds=int(rng.integers(0, 3600)))

def pick(roles=None):
    return random.choice([u for u in users if roles is None or u.role in roles])

def weekday_in(lo, hi):
    while True:
        d = int(rng.integers(lo, hi + 1))
        if (START + timedelta(days=d)).weekday() < 5:
            return d

def role_event(u, ts, ip, region=None):
    src, name, _ = random.choice(ROLE_ACTIONS[u.role])
    return make_event(u, ts, src, name, ip, region=region)

def tag(events, attack_type):
    for e in events:
        e["is_anomaly"], e["attack_type"] = 1, attack_type
    return events

def off_hours_burst(lo, hi):
    u = pick(); d = weekday_in(lo, hi)
    hour = random.choice([h for h in range(24) if not (u.start_hour <= h < u.end_hour)])
    base = ts_at(d, hour)
    return [role_event(u, base + timedelta(seconds=int(rng.integers(0, 3000))), random.choice(ip_pool[u.name]))
            for _ in range(int(rng.integers(25, 60)))]

def new_location(lo, hi):
    u = pick(); d = weekday_in(lo, hi); base = ts_at(d, int(rng.integers(u.start_hour, u.end_hour)))
    ip, region = random.choice(SUSPICIOUS_IPS), random.choice(EXOTIC_REGIONS)
    return [role_event(u, base + timedelta(seconds=int(rng.integers(0, 3000))), ip, region)
            for _ in range(int(rng.integers(6, 15)))]

def priv_esc(lo, hi):
    u = pick(["developer", "analyst", "devops"]); d = weekday_in(lo, hi)
    base = ts_at(d, int(rng.integers(u.start_hour, u.end_hour))); ip = random.choice(ip_pool[u.name])
    evs = []
    for i, n in enumerate(["CreateUser", "AttachUserPolicy", "CreateAccessKey"]):
        e = make_event(u, base + timedelta(seconds=40 * i), "iam.amazonaws.com", n, ip)
        if n == "AttachUserPolicy":
            e["requestParameters"] = {"policyArn": "arn:aws:iam::aws:policy/AdministratorAccess"}
        evs.append(e)
    return evs

def brute_force(lo, hi):
    u = pick(); d = weekday_in(lo, hi); base = ts_at(d, int(rng.integers(0, 24))); ip = random.choice(SUSPICIOUS_IPS)
    return [make_event(u, base + timedelta(seconds=int(rng.integers(0, 600))), "signin.amazonaws.com",
                       "ConsoleLogin", ip, error="Failed authentication")
            for _ in range(int(rng.integers(30, 80)))]

RECON = [("iam.amazonaws.com", n) for n in ["ListUsers", "ListRoles", "ListPolicies", "ListAccessKeys", "GetAccountAuthorizationDetails"]] + \
        [("ec2.amazonaws.com", n) for n in ["DescribeInstances", "DescribeSecurityGroups", "DescribeVpcs"]] + \
        [("s3.amazonaws.com", "ListBuckets")]

def recon_sweep(lo, hi):
    u = pick(); d = weekday_in(lo, hi); base = ts_at(d, int(rng.integers(u.start_hour, u.end_hour)))
    ip = random.choice(ip_pool[u.name])
    return [make_event(u, base + timedelta(seconds=int(rng.integers(0, 300))), *random.choice(RECON), ip)
            for _ in range(int(rng.integers(40, 80)))]

def cover_tracks(lo, hi):
    u = pick(["admin"]); d = weekday_in(lo, hi); base = ts_at(d, int(rng.integers(u.start_hour, u.end_hour)))
    ip = random.choice(ip_pool[u.name])
    return [make_event(u, base + timedelta(seconds=30 * i), "cloudtrail.amazonaws.com", n, ip)
            for i, n in enumerate(["StopLogging", "DeleteTrail"])]

def subtle_misuse(lo, hi):
    u = pick(); d = weekday_in(lo, hi); base = ts_at(d, int(rng.integers(u.start_hour, u.end_hour)))
    ip = random.choice(ip_pool[u.name])
    return [make_event(u, base + timedelta(seconds=int(rng.integers(0, 1200))), "secretsmanager.amazonaws.com",
                       "GetSecretValue", ip) for _ in range(int(rng.integers(4, 8)))]

TRAIN_DAYS, TEST_DAYS = (0, 20), (21, 29)
train_attacks = [off_hours_burst, new_location, priv_esc, brute_force]
held_out      = [recon_sweep, cover_tracks, subtle_misuse]

attacks = []
for fn in train_attacks:
    for _ in range(3): attacks += tag(fn(*TRAIN_DAYS), fn.__name__)
for fn in train_attacks + held_out:
    for _ in range(4): attacks += tag(fn(*TEST_DAYS), fn.__name__)

df = pd.concat([df_normal, pd.DataFrame(attacks)], ignore_index=True)
df = df.sort_values("eventTime").reset_index(drop=True)
df["user"] = df.userIdentity.apply(lambda x: x["userName"])
print(df.attack_type.value_counts())
print("event-level anomaly share:", round(df.is_anomaly.mean(), 4))

attack_type
normal             26947
brute_force          408
off_hours_burst      307
recon_sweep          230
new_location          81
subtle_misuse         24
priv_esc              21
cover_tracks           8
Name: count, dtype: int64
event-level anomaly share: 0.0385
